In [ ]:
# conda activate chronocell

import os, sys
import numpy as np
import pandas as pd

sys.path.append("/mnt/lareaulab/reliscu/programs/FGP_2024")
sys.path.append("/mnt/lareaulab/reliscu/projects/Chronocell/code")

import Chronocell

os.chdir("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/janssens_2025_preprint")

In [4]:
# Get traj object from running Chronocell
import pickle
with open("eLNPs_var>1.2_traj_WS.pkl", "rb") as f:
    traj = pickle.load(f)

In [5]:
def _get_state_per_time(tau, time_grid):
    time_states = np.zeros(shape=time_grid.shape, dtype=int) 
    for k in range(1, len(tau)):  
        idx = (time_grid <= tau[k]) & (time_grid > tau[k - 1])
        time_states[idx] = k - 1
    return time_states

In [ ]:
tau = traj.tau # State transition times (global)
time_grid = traj.t # Process time

thetas = traj.theta # Gene-specific parameters
alphas = thetas[:, 1:len(traj.topo.flatten())] # Transcription rates
betas = thetas[:,-2] # Splicing rates
gammas = thetas[:,-1] # Degradation rates


In [66]:
state_per_t = _get_state_per_time(tau, time_grid)
state_counts = np.bincount(state_per_t)
alphas_per_t = np.repeat(alphas, state_counts, axis=1)

In [ ]:
# cell_idx = 1

In [ ]:
posterior_prob = traj.Q.sum(1) # [:, 0, :] # Posterior probabilities of cell being in a given state
starting_state_inds = np.argmax(posterior_prob, axis=1) # Most likely state for each cell

In [98]:

lambda_U = np.empty((traj.X.shape[0], traj.X.shape[1], len(time_grid))) # Shape: cells x genes x time
lambda_S = lambda_U

# Transcriptional history starts with observed data:
cell_inds = np.arange(lambda_S.shape[0])
lambda_U[cell_inds, :, starting_state_inds] = traj.X[:, :, 0]
lambda_S[cell_inds, :, starting_state_inds] = traj.X[:, :, 1]

In [99]:
lambda_S[0, :, starting_state_idx[0]].shape

(662,)

In [ ]:

# # Start at current_state_idx position, and work backwards:

# for k in reversed(range(1, starting_state_indx + 1)): # range(1, len(time_dt))
    
#     # t_k = time_dt[k]  # Current time step (chronologically later)
#     # t_k1 = time_dt[k-1] # Previous time step (chronologically earlier)
#     # alpha_k1 = alphas_dt[k-1] # Previous transcription rate (chronologically later)

#     t_k = time_grid[k]
#     t_k1 = time_grid[k-1]
#     alpha_k1 = alpha_per_t[k-1]
    
#     # Solve for previous time point
#     lambda_u_k1 = (lambda_u[-1] - (alpha_k1 / betas) * (1 - np.exp(-betas * (t_k - t_k1)))) / np.exp(-beta * (t_k - t_k1))
#     lambda_u_k1 = max(lambda_u_k1, 0) # No. RNAs can't be negative
#     lambda_s_k1 = (lambda_s[-1] - (betas / (betas - gammas)) * lambda_u_k1 + (beta * alpha_k1) /
#                     (gammas * (betas - gammas)) * (1 - np.exp(-gammas * (t_k - t_k1)))) / np.exp(-gammas * (t_k - t_k1))
#     lambda_s_k1 = max(lambda_s_k1, 0) # No. RNAs can't be negative 
        
#     # lambda_u.append(lambda_u_k1)
#     # lambda_s.append(lambda_s_k1)
#     lambda_U[:, :, k] = lambda_u_k1
#     lambda_S[:, :, k] = lambda_s_k1

In [133]:
n_cells, n_genes, m = lambda_U.shape
dt = np.diff(time_grid)

In [ ]:

for k in reversed(range(1, m)):
    mask = starting_state_inds >= k
    if not np.any(mask):
        continue
    dt_k = dt[k-1]

    lambda_u_k = lambda_U[mask, :, k]  
    lambda_s_k = lambda_S[mask, :, k]

    alphas_k1 = alphas_per_t[:, k-1] # Previous time points' transcription rates

    # Solve for previous time point
    exp_b = np.exp(-betas * dt_k)
    exp_g = np.exp(-gammas * dt_k)

    lambda_u_k1 = (lambda_u_k - (alphas_k1 / betas) * (1 - exp_b)) / exp_b
    lambda_u_k1 = np.max(lambda_u_k1, 0, axis=1) # No. RNAs can't be negative

    lambda_s_k1 = (
        lambda_s_k 
            - (betas / (betas - gammas)) * lambda_u_k1 
            + (betas * alphas_k1) / (gammas * (betas - gammas)) * (1 - exp_g)
        ) / exp_g
    lambda_s_k1 = max(lambda_s_k1, 0) 

    lambda_U[mask, :, k-1] = lambda_u_k1
    lambda_S[mask, :, k-1] = lambda_s_k1

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [138]:
k

99

In [ ]:
lambda_u_k1

(1490, 662)